In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import stats
import math
import glob
import pandas as pd

In [ ]:
#Salvador - color invariants (c1, c2, c3)

from scipy import stats

folder = "images_MP"
files = glob.glob(folder+"/*.png")
files.sort()
print (files)

coef_S = 1.3
coef_H = 1


df = pd.DataFrame() 
idx = 0

for f in files:
    
    index = 1
    
    img = cv2.imread(f)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print (f)
        
    size = img.shape
    img_invar = np.zeros((size[0], size[1], 3))
        
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = int(img_rgb[i,j,0])
            G = int(img_rgb[i,j,1])
            B = int(img_rgb[i,j,2])
            
            RGB = (R,G,B)
            #print (RGB)
            
            c1 = np.arctan(R/max(G,B))
            c2 = np.arctan(G/max(R,B))
            c3 = np.arctan(B/max(R,G))
            
            img_invar[i,j,2] = c1
            img_invar[i,j,1] = c2
            img_invar[i,j,0] = c3

    if "287" in f:
        colors = ([302, 344], [626, 305], [493, 254], [507, 253], [627, 265])

    elif "307" in f:
        colors = ([160, 653], [252, 817], [220, 712], [219, 584], [138, 659], [302, 852], [165, 710],[242, 570])

    elif "305" in f:
        colors = ([466, 276], [400, 313], [519, 326], [1013, 521], [420, 367])  

    elif "291" in f:
        colors = ([526, 708], [531, 394], [420, 607], [380, 475], [450, 707])
    
    for c in colors:
                
        x = c[0]
        y = c[1]
        
        winsz = 8   #ajustar en f(x) del soroll/textura

        color_roi = (255,0,0)
        
        #img_roi = []
        
        img = cv2.imread(f)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_roi = cv2.rectangle(img_rgb, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)

        c1_roi_np = img_invar[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]
        #h_circ_np = (h_roi_np)*2
        
        c1_roi = c1_roi_np.flatten()
        c1_mean = np.mean(c1_roi)
        c1_std = np.std(c1_roi)
        #print (h_roi)

        min_c1 = np.min(c1_roi)
        max_c1 = np.max(c1_roi)
        print (min_c1,   max_c1)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((c1_roi > (c1_mean+k*c1_std)) | (c1_roi < (c1_mean-k*c1_std)))[0]
        print ("c1 outliers:",  outlier_indices)
        c1_no_out = np.delete(c1_roi, outlier_indices)
        
        c1 = np.mean(c1_no_out)
        min_c1 = np.min(c1_no_out)
        max_c1 = np.max(c1_no_out)
        c1_noise = max_c1-min_c1
        #std_h = stats.circstats.circstd(h_circ, method='angular')
        print ("c1: ", c1)
        

        c2_roi_np = img_invar[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
       #Descartar outliers amb mètode Chevyshev
        c2_roi = c2_roi_np.flatten()
        c2_mean = np.mean(c2_roi)
        c2_std = np.std(c2_roi)
        #print (h_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((c2_roi > (c2_mean+k*c2_std)) | (c2_roi < (c2_mean-k*c2_std)))[0]
        print ("c2 outliers:",  outlier_indices)
        c2_no_out = np.delete(c2_roi, outlier_indices)

        c2 = np.mean(c2_no_out)
        std_c2 = np.std(c2_no_out)
        min_c2 = np.min(c2_no_out)
        max_c2 = np.max(c2_no_out)
        c2_noise = max_c2-min_c2 
        print ("c2: ", c2)


        c3_roi_np = img_invar[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        
       #Descartar outliers amb mètode Chevyshev
        c3_roi = c3_roi_np.flatten()
        c3_mean = np.mean(c3_roi)
        c3_std = np.std(c3_roi)
        #print (h_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((c3_roi > (c3_mean+k*c3_std)) | (c3_roi < (c3_mean-k*c3_std)))[0]
        print ("c3 outliers:",  outlier_indices)
        c3_no_out = np.delete(c3_roi, outlier_indices)

        c3 = np.mean(c3_no_out)
        std_c3 = np.std(c3_no_out)
        min_c3 = np.min(c3_no_out)
        max_c3 = np.max(c3_no_out)
        c3_noise = max_c3-min_c3 
        print ("c3: ", c3)

    
        limit1 = (min_c1, min_c2, min_c3) 
        limit2 = (max_c1, max_c2, max_c3)
    
        print(limit1,limit2)

        mask = cv2.inRange(img_invar, limit1, limit2)
        result = cv2.bitwise_and(img, img, mask=mask)
        result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        result_name = "../results/"+f[-13:-10]+"_Salvador_"+str(winsz)+"_"+str(index)+".png"
        cv2.imwrite (result_name, result)


        img_roi_rgb = cv2.cvtColor(img_roi, cv2.COLOR_RGB2BGR)
        img_roi_name = "../results/"+f[-13:-10]+"ROI"+str(index)+".png"
        cv2.imwrite (img_roi_name, img_roi_rgb)

        df.at[idx, "Image"] = f
        df.at[idx, "color"] = str(index)
        df.at[idx, "c1"] = c1
        df.at[idx, "c2"] = c2
        df.at[idx, "c3"] = c3
        df.at[idx, "c1 noise"] = c1_noise
        df.at[idx, "c2 noise"] = c2_noise
        df.at[idx, "c3 noise"] = c3_noise


        idx += 1
        
        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 2, 1)
        plt.imshow(img_roi)
        plt.subplot(1,2,2)
        plt.imshow(result_bgr)
        plt.show()
        
    
        index += 1

print(df)
df.to_excel("../results/Salvador_results.xlsx") 